# Train the PII extractor: Qwen3-1.7B + LoRA (Google Colab)

This notebook builds the training data, trains a LoRA adapter on `Qwen/Qwen3-1.7B`, publishes the adapter on the Hugging Face Hub and exports a GGUF file for Ollama, so the tuned model runs locally in the app.

**Before you start**

1. The golden test (`data/golden_test.jsonl`) is written, double-checked and **committed to GitHub**. This notebook refuses to continue without it, and prints the commit date as evidence.
2. Runtime > Change runtime type > **T4 GPU**.
3. You have a Hugging Face account and a token with write access.

**Status of this notebook:** it was written without access to a GPU and has not been executed end to end. The pinned versions in `requirements-train.txt` were the latest on PyPI on 2026-11-15 and were not tested together. Run it cell by cell the first time and read each output. No number produced here is a project result; results come from `python -m src.evaluate` on your own machine.

## 1. Settings

In [ ]:
# Set these two values, then run the cell.
REPO_URL = ""      # your team's GitHub repository, e.g. "https://github.com/<team>/<repo>.git"
HF_REPO_ID = ""    # where the adapter is published, e.g. "<hf-user>/pii-qwen3-1.7b-lora"

BASE_MODEL = "Qwen/Qwen3-1.7B"   # 1.7B parameters: inside the 4B limit of the project
GGUF_NAME = "pii-qwen3-1.7b-q8_0.gguf"

USE_QLORA = False   # True: load the base model in 4-bit (less GPU memory, slightly slower)
EPOCHS = 2
LEARNING_RATE = 1e-4
MAX_LENGTH = 768    # tokens per training example (system prompt + sentence + answer)
BATCH_SIZE = 4
GRAD_ACCUMULATION = 4
LORA_R = 16
LORA_ALPHA = 32
SEED = 13

assert REPO_URL and HF_REPO_ID, "Fill in REPO_URL and HF_REPO_ID above."

## 2. GPU, code and libraries

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU."
print(torch.cuda.get_device_name(0), "| torch", torch.__version__)

In [ ]:
!git clone {REPO_URL} repo
%cd repo
!pip install -q -r requirements-train.txt

## 3. The golden test comes first

The first line printed is the last commit that touched the golden test. Its date must be earlier than this training run. The validator then checks the format and that every row was written by one member and checked by another.

In [ ]:
!git log -1 --format="golden test last commit: %h  %cI  %s" -- data/golden_test.jsonl
!python -m src.validate_golden data/golden_test.jsonl

## 4. Build the training data

`src.data_prep` converts ANERCorp (downloaded from the Hub), any Wojood files placed in `data/raw/wojood/`, any team-written files named `data/team_*.jsonl`, and seeded template sentences into chat-format JSONL. It removes every sentence that equals, nearly equals or contains a golden sentence and fails if any overlap remains. Keep `data/processed/prep_report.json`: it is the proof for the dataset card.

In [ ]:
import glob, subprocess, sys

cmd = [sys.executable, "-m", "src.data_prep",
       "--golden", "data/golden_test.jsonl",
       "--anercorp-hf", "asas-ai/ANERCorp",
       "--synthetic-n", "600",
       "--seed", str(SEED),
       "--out", "data/processed"]
wojood_files = sorted(glob.glob("data/raw/wojood/*.txt"))
team_files = sorted(glob.glob("data/team_*.jsonl"))
if wojood_files:
    cmd += ["--wojood", *wojood_files]
if team_files:
    cmd += ["--team-jsonl", *team_files]
print(" ".join(cmd))
subprocess.run(cmd, check=True)
print(open("data/processed/prep_report.json", encoding="utf-8").read()[:1500])

## 5. Load the data and look at what the model will see

Compare the two printouts. The training example should contain the same turn markers as the inference prompt, including the empty `<think>` block that Qwen3 uses when thinking is off. If the training text has no `<think>` block while the inference prompt has one, tell the team before training: the model would be trained in a slightly different format from the one it is served in.

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

ds = load_dataset("json", data_files={"train": "data/processed/train.jsonl",
                                      "validation": "data/processed/val.jsonl"})
ds = ds.select_columns(["messages"])
print(ds)

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
example = ds["train"][0]["messages"]
print("=== one training example as the model sees it ===")
print(tok.apply_chat_template(example, tokenize=False))
print("=== the prompt used at inference (thinking off) ===")
print(tok.apply_chat_template(example[:-1], tokenize=False, add_generation_prompt=True, enable_thinking=False))

sample = ds["train"].select(range(min(500, len(ds["train"]))))["messages"]
lengths = [len(tok(tok.apply_chat_template(m, tokenize=False))["input_ids"]) for m in sample]
too_long = sum(n > MAX_LENGTH for n in lengths) / len(lengths)
print(f"token length: max {max(lengths)}, share above MAX_LENGTH: {too_long:.1%}")

## 6. Train the LoRA adapter

`assistant_only_loss=True` puts the loss on the JSON answer only, not on the instruction or the input sentence. If the GPU runs out of memory, set `USE_QLORA = True` in the settings cell, or lower `BATCH_SIZE` and raise `GRAD_ACCUMULATION` by the same factor, then restart the runtime.

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

if USE_QLORA:
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig
    model_or_id = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True),
        device_map="auto",
    )
else:
    model_or_id = BASE_MODEL

peft_config = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05,
                         target_modules="all-linear", task_type="CAUSAL_LM")

training_args = SFTConfig(
    output_dir="outputs/pii-qwen3-lora",
    assistant_only_loss=True,
    max_length=MAX_LENGTH,
    packing=False,
    learning_rate=LEARNING_RATE,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUMULATION,
    lr_scheduler_type="cosine",
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=True,
    seed=SEED,
    report_to="none",
    hub_model_id=HF_REPO_ID,
)

trainer = SFTTrainer(
    model=model_or_id,
    args=training_args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    peft_config=peft_config,
)
trainer.model.print_trainable_parameters()
trainer.train()

## 7. Save the adapter and confirm it is an adapter

In [ ]:
import os

ADAPTER_DIR = "outputs/pii-qwen3-lora/final_adapter"
trainer.save_model(ADAPTER_DIR)
tok.save_pretrained(ADAPTER_DIR)
files = sorted(os.listdir(ADAPTER_DIR))
print(files)
assert "adapter_config.json" in files, "Expected a LoRA adapter (adapter_config.json), not full model weights."

## 8. Smoke check on a few validation sentences

This only shows that the tuned model answers in the expected JSON format. It is not the evaluation: the validation split comes from the training sources, not from the golden test.

In [ ]:
from src.common import ModelOutputError, parse_model_output

model = trainer.model
model.eval()
for row in ds["validation"].select(range(min(5, len(ds["validation"])))):
    messages = row["messages"]
    prompt = tok.apply_chat_template(messages[:-1], tokenize=False,
                                     add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
    reply = tok.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    try:
        parsed = parse_model_output(reply)
        verdict = f"valid JSON, {len(parsed.entities)} entities" + (" (needed cleaning)" if parsed.cleaned else "")
    except ModelOutputError as exc:
        verdict = f"INVALID: {exc}"
    print("TEXT    :", messages[1]["content"])
    print("EXPECTED:", messages[2]["content"])
    print("MODEL   :", reply)
    print("        ->", verdict, "\n")

## 9. Publish the adapter and its model card on the Hub

`login()` asks for your token. Do not paste the token into a cell and do not commit it. Fill in `docs/MODEL_CARD.md` with your real evaluation results once you have them and upload it again; until then the card says the results are pending.

In [ ]:
from huggingface_hub import HfApi, login

login()
api = HfApi()
api.create_repo(HF_REPO_ID, exist_ok=True)
api.upload_folder(folder_path=ADAPTER_DIR, repo_id=HF_REPO_ID)
api.upload_file(path_or_fileobj="docs/MODEL_CARD.md", path_in_repo="README.md", repo_id=HF_REPO_ID)
print("published: https://huggingface.co/" + HF_REPO_ID)

## 10. Merge the adapter and export a GGUF file for Ollama

Ollama runs GGUF files. The adapter is merged into a full-precision copy of the base model and converted with the script from the llama.cpp repository (`q8_0`: 8-bit weights, about 1.8 GB for this model). The published artefact that the project requires is still the adapter from step 9; the GGUF file is only how it is served locally.

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM

del trainer, model
torch.cuda.empty_cache()

base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.float16)
merged = PeftModel.from_pretrained(base, ADAPTER_DIR).merge_and_unload()
MERGED_DIR = "outputs/merged"
merged.save_pretrained(MERGED_DIR)
tok.save_pretrained(MERGED_DIR)
print(sorted(os.listdir(MERGED_DIR)))

In [ ]:
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!pip install -q sentencepiece protobuf
!python /content/llama.cpp/convert_hf_to_gguf.py outputs/merged --outfile outputs/{GGUF_NAME} --outtype q8_0
!ls -lh outputs/{GGUF_NAME}

In [ ]:
# Store the GGUF next to the adapter on the Hub so every team member can download it.
api.upload_file(path_or_fileobj=f"outputs/{GGUF_NAME}", path_in_repo=f"gguf/{GGUF_NAME}", repo_id=HF_REPO_ID)
print(f"download from: https://huggingface.co/{HF_REPO_ID}/tree/main/gguf")

## 11. Record the exact library versions

Commit `requirements-train.lock.txt` to the repository. It lists the versions that actually ran, which is what "pinned versions" has to mean for the training code.

In [ ]:
!pip freeze > requirements-train.lock.txt
!wc -l requirements-train.lock.txt

## 12. Back on your own machine

```bash
# 1. Download gguf/pii-qwen3-1.7b-q8_0.gguf from the Hub repository's Files tab into models/
# 2. Create the Ollama model (reuses the chat template of the base model)
ollama pull qwen3:1.7b
python scripts/make_modelfile.py --gguf models/pii-qwen3-1.7b-q8_0.gguf
ollama create pii-qwen3-lora -f Modelfile

# 3. Evaluate the three systems on the golden test, then run the safety check
python -m src.evaluate --base-model qwen3:1.7b --tuned-model pii-qwen3-lora
python -m src.safety_check --model pii-qwen3-lora

# 4. Demo
streamlit run app/streamlit_app.py
```